# Fig. S26 | Recovery timing

Plots changes in half-recovery timing.

In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'management').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from src.management.experiments import WINDOW_2013, context
DATA=ROOT/'outputs'/'MANAGEMENT_2012_2013'/'equal_volume'
CACHE=ROOT/'outputs'/'MANAGEMENT_2012_2013'/'_cache'/'recovery_responses'
OUT=ROOT/'outputs'/'figures'/'FigS26'/'FigS26.png'
OUT.parent.mkdir(parents=True,exist_ok=True)
cells=pd.read_csv(DATA/'recovery_time_cells.csv.gz',dtype={'seed':str})
PI75=1.150349
mpl.rcParams.update({'font.family':'Arial','font.size':10.5,'axes.labelsize':12.5,'axes.titlesize':12.5,'xtick.labelsize':9.5,'ytick.labelsize':10,'axes.linewidth':0.8,'axes.spines.top':True,'axes.spines.right':True})

In [ ]:
eligible=cells[cells.valid_flag & ~cells.reached_before_intervention]
comparable=eligible[np.isfinite(eligible.delta_T50)]
thresholds=np.array([1,6,12,24,36,60])
curves=[]
for _,group in comparable.groupby('seed'):
    gains=group.loc[group.delta_T50>0,'delta_T50'].to_numpy(float)
    curves.append([np.mean(gains>=threshold) for threshold in thresholds])
curves=np.asarray(curves); mean=100*curves.mean(0); radius=100*PI75*curves.std(0,ddof=0)
fig,axes=plt.subplots(1,3,figsize=(9.6,3.25))
x=np.arange(len(thresholds)); axes[0].bar(x,mean,width=0.68,color='#486A9A',edgecolor='#365579',lw=0.5)
axes[0].errorbar(x,mean,yerr=radius,fmt='none',color='#365579',capsize=2,lw=0.9)
axes[0].set_xticks(x,thresholds); axes[0].set_ylim(0,105); axes[0].set(xlabel='Time gained (months)',ylabel='Improved Cells (%)',title='a  Uniform recovery-time gains')

ctx=context()
dates=pd.date_range('2013-05-01','2014-04-01',freq='MS')
date_labels=np.asarray([d.strftime('%Y-%m') for d in dates])
strategy_files={'Uniform':'uniform','Leverage-guided':'leverage'}
strategy_colors={'Historical pumping':'0.25','Uniform':'#486A9A','Leverage-guided':'#16857C'}
strategy_styles={'Historical pumping':'-','Uniform':'--','Leverage-guided':'-'}
strategy_markers={'Uniform':'o','Leverage-guided':'s'}

def labels_from_index(values):
    labels=np.full(len(values),'',dtype=object)
    finite=np.isfinite(values)
    labels[finite]=ctx.months[values[finite].astype(int)]
    return labels

def cumulative_mean_curves(class_mask):
    managed_by_strategy={name:[] for name in strategy_files}
    baseline_curve=None
    denominator=None
    for name,stem in strategy_files.items():
        for seed in ('seed11','seed22','seed33','seed44','seed55'):
            with np.load(CACHE/f'{stem}_20_{seed}.npz') as cached:
                response=cached['response_m'][:len(date_labels)]
                arrays=ctx.recovery_arrays(response,WINDOW_2013)
            eligible_mask=arrays['valid'] & ~arrays['reached_before_intervention'] & class_mask
            if denominator is None:
                denominator=int(eligible_mask.sum())
                base_labels=labels_from_index(arrays['baseline_t50_index'][eligible_mask])
                baseline_curve=np.array([np.mean((base_labels!='') & (base_labels<=month)) for month in date_labels])
            elif int(eligible_mask.sum()) != denominator:
                raise AssertionError('Eligible-cell denominator changed across strategies or seeds.')
            managed_labels=labels_from_index(arrays['management_t50_index'][eligible_mask])
            managed_by_strategy[name].append([np.mean((managed_labels!='') & (managed_labels<=month)) for month in date_labels])
    means={name:np.asarray(curves,float).mean(axis=0) for name,curves in managed_by_strategy.items()}
    return baseline_curve,means,denominator

all_mask=np.ones(len(ctx.grid_ids),dtype=bool)
slow_mask=ctx.response_class=='Slow recovery'
for ax,class_mask,title in [(axes[1],all_mask,'b  All eligible Cells'),(axes[2],slow_mask,'c  Slow-recovery Cells')]:
    base,means,denominator=cumulative_mean_curves(class_mask)
    ax.plot(dates,100*base,color=strategy_colors['Historical pumping'],ls=strategy_styles['Historical pumping'],lw=1.7,label='Historical pumping')
    for name in ('Uniform','Leverage-guided'):
        ax.plot(dates,100*means[name],color=strategy_colors[name],ls=strategy_styles[name],lw=1.7,label=name,marker=strategy_markers[name],markevery=[-1],ms=3.2)
    ax.set(xlabel='Time',ylabel=r'Cells reaching $D_{50}$ (%)',title=title,ylim=(0,45),xlim=(dates.min(),dates.max()))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
axes[1].legend(frameon=False,fontsize=7.2,loc='upper left')
axes[1].text(0.97,0.055,'Earlier than historical by Apr 2014\nUniform 3.77%; Leverage-guided 3.19%',transform=axes[1].transAxes,ha='right',va='bottom',fontsize=6.8,color='0.25')
for ax in axes:
    ax.title.set_fontweight('bold'); ax.title.set_ha('left'); ax.title.set_position((0,1)); ax.grid(axis='y',color='#ECE9E4',lw=0.5)
fig.tight_layout(w_pad=1.0); fig.savefig(OUT,dpi=600,bbox_inches='tight',facecolor='white'); plt.show()